In [2]:
import random
import functools

# Global log container for pipeline execution tracking
pipeline_execution_log = []

def monitor_decorator(func):
    """Requirement 5: Decorator to monitor major processing functions."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        log_entry = f"[LOG] Started execution of process '{func.__name__}'."
        pipeline_execution_log.append(log_entry)
        result = func(*args, **kwargs)
        pipeline_execution_log.append(f"[LOG] Finished execution of process '{func.__name__}'.")
        return result
    return wrapper

def generate_telemetry_stream(surname: str, seed_digit: int, artist: str, length: int = 8):
    """Requirements 1 & 3: Generator function yielding continuous student-specific telemetry stream."""
    seed_val = sum(ord(c) for c in surname) + seed_digit + sum(ord(c) for c in artist)
    random.seed(seed_val)
    
    # Stream contains valid floats, out-of-bounds readings, and corrupt text entries
    raw_stream = [72.5, 125.0, "ERR_SENSOR", 45.0, 95.0, -10.0, "MALFUNCTION", 88.0]
    for reading in raw_stream[:length]:
        yield reading

def analyze_abnormal_recursive(abnormal_list: list, index: int = 0) -> list:
    """Requirement 7: Recursive function to trace detected abnormal conditions down to base case."""
    if index >= len(abnormal_list):
        pipeline_execution_log.append(f"[LOG] Recursive trace reached base condition (end of list at depth {index}).")
        return [f"Base condition reached at depth {index}."]
    
    log_msg = f"Tracing abnormal value {abnormal_list[index]} at depth {index + 1}"
    pipeline_execution_log.append(f"[LOG] {log_msg}")
    return [log_msg] + analyze_abnormal_recursive(abnormal_list, index + 1)

In [ ]:
import types

# Use functions already defined in the notebook instead of a missing module.
tu = types.SimpleNamespace(
    monitor_decorator=monitor_decorator,
    generate_telemetry_stream=generate_telemetry_stream,
    analyze_abnormal_recursive=analyze_abnormal_recursive,
    pipeline_execution_log=pipeline_execution_log,
)

LAST_NAME = "LINSANGAN"
SEED_NUM = 6
FAVORITE_ARTIST = "STEVE PERRY"

@tu.monitor_decorator
def run_monitoring_pipeline(surname: str, seed_digit: int, artist: str):
    """Main processing pipeline handling telemetry stream validation, transformation, and analysis."""
    stream = tu.generate_telemetry_stream(surname, seed_digit, artist)
    
    processed_count = 0
    valid_readings = []
    invalid_readings = []
    abnormal_conditions = []

    # Requirement 4: Lambda function to transform/calibrate telemetry values
    calibrate_reading = lambda val: round(val * 1.02, 2)

    # Process stream continuous items without storing whole stream upfront
    for raw_val in stream:
        processed_count += 1
        
        # Requirement 6: Exception handling for invalid or unexpected values
        try:
            if not isinstance(raw_val, (int, float)):
                raise TypeError(f"Invalid telemetry type: '{raw_val}'")
            if raw_val < 0 or raw_val > 100:
                raise ValueError(f"Value {raw_val} out of valid operating bounds (0-100)")

            calibrated = calibrate_reading(raw_val)
            valid_readings.append(calibrated)

            if calibrated > 90.0:
                abnormal_conditions.append(calibrated)
                tu.pipeline_execution_log.append(f"[LOG] Abnormal condition detected: {calibrated}")

        except (TypeError, ValueError) as e:
            invalid_readings.append(f"Reading #{processed_count} ({raw_val}): {e}")
            tu.pipeline_execution_log.append(f"[LOG] Exception caught on reading #{processed_count}: {e}")

    # Requirement 7: Call recursive analysis on abnormal values
    recursive_logs = tu.analyze_abnormal_recursive(abnormal_conditions)

    # Determine overall status
    if len(abnormal_conditions) > 1 or len(invalid_readings) >= 4:
        overall_status = "CRITICAL - IMMEDIATE INSPECTION REQUIRED"
    elif len(abnormal_conditions) == 1:
        overall_status = "WARNING - ELEVATED READINGS DETECTED"
    else:
        overall_status = "OPTIMAL"

    # Requirement 8: Return structured dictionary report
    return {
        "inputs": f"{surname}_{seed_digit}_{artist}",
        "processed_count": processed_count,
        "valid_readings": valid_readings,
        "invalid_readings": invalid_readings,
        "abnormal_conditions": abnormal_conditions,
        "recursive_analysis": recursive_logs,
        "overall_status": overall_status
    }

if __name__ == "__main__":
    report = run_monitoring_pipeline(LAST_NAME, SEED_NUM, FAVORITE_ARTIST)
    print(report)

{'inputs': 'SMITH_7_COLDPLAY', 'processed_count': 8, 'valid_readings': [73.95, 45.9, 96.9, 89.76], 'invalid_readings': ['Reading #2 (125.0): Value 125.0 out of valid operating bounds (0-100)', "Reading #3 (ERR_SENSOR): Invalid telemetry type: 'ERR_SENSOR'", 'Reading #6 (-10.0): Value -10.0 out of valid operating bounds (0-100)', "Reading #7 (MALFUNCTION): Invalid telemetry type: 'MALFUNCTION'"], 'abnormal_conditions': [96.9], 'recursive_analysis': ['Tracing abnormal value 96.9 at depth 1', 'Base condition reached at depth 1.'], 'overall_status': 'CRITICAL - IMMEDIATE INSPECTION REQUIRED'}
